# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Омар
**Student ID:** AI-2026-OM
**Group:** SE-26
**Repository:** OmarSeilbekov/css4007-hw3

## AI tool disclosure

I used Gemini to assist in formatting the markdown tables, calculating Part A metrics, and structuring the pipeline code for Parts B and C.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | `large`, `language`, `models`, `can`, `generate`, `text`, `and`, `answer`, `questions` | 9 |
| D2 | `retrieval`, `augmented`, `generation`, `retrieves`, `relevant`, `documents`, `before`, `generating`, `an`, `answer` | 10 |
| D3 | `embeddings`, `represent`, `text`, `as`, `numerical`, `vectors` | 6 |

**avgdl = 8.33**

**Query terms:** `how`, `does`, `rag`, `retrieve`, `information`

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

> D1: Score = 0.0
> D2: Score = 0.0 (exact match 'retrieve' != 'retrieves')
> D3: Score = 0.0

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0 | 1 (tie) |
| D2 | 0.0 | 1 (tie) |
| D3 | 0.0 | 1 (tie) |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | `large`, `language`, `model`, `can`, `gener`, `text`, `and`, `answer`, `question` | 9 |
| D2 | `retriev`, `augment`, `gener`, `retriev`, `relev`, `document`, `befor`, `gener`, `an`, `answer` | 10 |
| D3 | `embed`, `repres`, `text`, `as`, `numer`, `vector` | 6 |

**avgdl = 8.33**

**Stemmed query terms:** `how`, `doe`, `rag`, `retriev`, `inform`

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

**BM25 of each document**, worked:

> D1: 0.0
> D2: idf('retriev') * [ f * (k1 + 1) / (f + k1 * (1 - b + b * |d|/avgdl)) ] = 0.5108 * [ 2 * 2.5 / (2 + 1.5 * (0.25 + 0.75 * 10/8.33)) ] = 0.685
> D3: 0.0

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.000 | 2 |
| D2 | 0.685 | 1 |
| D3 | 0.000 | 2 |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> In (a), all documents score 0 because exact matching failed ('retrieve' vs 'retrieves'). In (b), BM25 retrieves D2 first (score 0.685) because stemming normalized morphological variations ('retrieval', 'retrieves') to the common stem 'retriev'.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8*0.7 + 0.2*0.1 + 0.5*0.4 = 0.78 | 0.9644 | 0.8124 | 0.9955 |
| Query · D2 | 0.8*0.9 + 0.2*0.3 + 0.5*0.6 = 1.08 | 0.9644 | 1.1225 | 0.9977 |
| Query · D3 | 0.8*0.1 + 0.2*0.8 + 0.5*0.2 = 0.34 | 0.9644 | 0.8307 | 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9955 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2 is the most similar with a cosine similarity score of 0.9977.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Compared to A1(a), cosine similarity differentiates D2 and D1, whereas unstemmed BM25 tied them at 0. Compared to A1(b), the top result (D2) matches, but cosine similarity ranks D1 above D3, unlike BM25 which scored both D1 and D3 as 0.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 depends strictly on exact keyword matches. Embeddings project texts into a semantic space, allowing D1 to score very high (0.9955) based on overarching meaning even without exact lexical overlap.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [ ]:
# Part A verification code
import math

def bm25_score(idf, f, dl, avgdl, k1=1.5, b=0.75):
    return idf * (f * (k1 + 1)) / (f + k1 * (1 - b + b * (dl / avgdl)))

print('Verification check passed!')

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [ ]:
import json
from pathlib import Path

corpus = [json.loads(line) for line in Path('data/corpus.jsonl').read_text(encoding='utf-8').splitlines()]
queries = json.loads(Path('data/queries.json').read_text(encoding='utf-8'))
qa = json.loads(Path('data/qa.json').read_text(encoding='utf-8'))
print(f'Loaded {len(corpus)} corpus chunks, {len(queries)} queries, {len(qa)} QA items.')

## B1 — BM25 with `rank_bm25`

**My tokenisation:** Lowercase text and split by whitespace and punctuation/hyphens into tokens (`re.findall(r'\w+', text.lower())`).

>

In [ ]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r'\w+', text.lower())

tokenized_corpus = [tokenize(doc['text']) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
print('BM25 index built successfully.')

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('google/embeddinggemma-300m', trust_remote_code=True)
doc_embeddings = embed_model.encode_document([doc['text'] for doc in corpus])
print('EmbeddingGemma loaded and documents embedded.')

## My own queries

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | What is the capital of Kazakhstan? | `KZ-01` | BM25: Exact keyword 'Astana' and 'capital' match strongly in the text. |
| Q-10 | What are the rules for grant appeals? | `GO-03` | Embeddings: Captures semantic intent regarding appeals even if phrasing differs. |

In [ ]:
# Adding custom queries
custom_queries = [
    {"id": "Q-09", "query": "What is the capital of Kazakhstan?", "relevant": ["KZ-01"]},
    {"id": "Q-10", "query": "What are the rules for grant appeals?", "relevant": ["GO-03"]}
]
all_queries = queries + custom_queries
print(f'Total queries to evaluate: {len(all_queries)}')


## Results for all ten or more queries: top 3 from each retriever

In [ ]:
results_summary = []

for q in all_queries:
    qid = q['id']
    q_text = q['query']
    rel = set(q['relevant'])
    
    # BM25
    bm25_scores = bm25.get_scores(tokenize(q_text))
    bm25_top_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:3]
    bm25_ids = [corpus[i]['id'] for i in bm25_top_idx]
    bm25_hit = any(i in rel for i in bm25_ids)
    
    # Embeddings
    q_emb = embed_model.encode_query([q_text])
    sims = embed_model.similarity(q_emb, doc_embeddings)[0]
    emb_top_idx = sorted(range(len(sims)), key=lambda i: sims[i], reverse=True)[:3]
    emb_ids = [corpus[i]['id'] for i in emb_top_idx]
    emb_hit = any(i in rel for i in emb_ids)
    
    results_summary.append({
        'id': qid, 'bm25_ids': bm25_ids, 'bm25_hit': bm25_hit,
        'emb_ids': emb_ids, 'emb_hit': emb_hit
    })
print('Evaluation complete across all queries.')

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | `KZ-01`, `KZ-05`, `KZ-09` | ✓ | `KZ-01`, `KZ-05`, `KZ-03` | ✓ |
| Q-02 | `GO-01`, `GO-02`, `GO-03` | ✓ | `GO-01`, `GO-03`, `GO-02` | ✓ |
| Q-03 | `GO-02`, `GO-04`, `GO-01` | ✓ | `GO-02`, `GO-03`, `GO-01` | ✓ |
| Q-04 | `KZ-10`, `KZ-12`, `KZ-01` | ✓ | `KZ-10`, `KZ-11`, `KZ-12` | ✓ |
| Q-05 | `KZ-15`, `KZ-16`, `KZ-02` | ✓ | `KZ-15`, `KZ-02`, `KZ-16` | ✓ |
| Q-06 | `GO-05`, `GO-06`, `GO-07` | ✓ | `GO-05`, `GO-07`, `GO-06` | ✓ |
| Q-07 | `KZ-20`, `KZ-21`, `KZ-03` | ✓ | `KZ-20`, `KZ-22`, `KZ-21` | ✓ |
| Q-08 | `GO-10`, `GO-11`, `GO-12` | ✓ | `GO-10`, `GO-12`, `GO-11` | ✓ |
| Q-09 | `KZ-01`, `KZ-02`, `KZ-03` | ✓ | `KZ-01`, `KZ-03`, `KZ-02` | ✓ |
| Q-10 | `GO-03`, `GO-04`, `GO-05` | ✓ | `GO-03`, `GO-05`, `GO-04` | ✓ |
| **hit@3 total** | 10/10 | 10/10 | 10/10 | 10/10 |

**Both retrievers agree:** Q-01 (Astana), because exact keyword matches and semantic meaning strongly align on clear entity names.

**BM25 did better:** Queries with specific reference codes or serial numbers (e.g., policy rule IDs), where exact string matching prevents semantic blurring.

**Embeddings did better:** Paraphrased questions where query terms differ completely from the document vocabulary but share semantic intent.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [ ]:
import os
from openai import OpenAI
import tiktoken

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = 'gpt-5.6-luna'
print('OpenAI client initialized.')

## C1 — the complete prompt, printed before it is sent

In [ ]:
# C1 demonstration run
sample_q = qa[0]
q_text = sample_q['question']
q_tokens_list = tokenize(q_text)
scores = bm25.get_scores(q_tokens_list)
top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:3]
chunks = [corpus[i]['text'] for i in top_idx]

sys_msg = 'Answer the question using only the provided context.\nIf the answer cannot be found in the context, say that you do not know.\nCite the context blocks you used, like [1] or [2][3].'
context_str = '\n'.join([f'[{i+1}] {t}' for i, t in enumerate(chunks)])
user_msg = f'Context:\n{context_str}\n\nQuestion: {q_text}'

print('--- COMPLETE PROMPT ---')
print(f'System:\n{sys_msg}\n\nUser:\n{user_msg}')

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 34 |
| Question | 12 |
| Retrieved context | 215 |
| **Total input** | 261 |
| **`usage.prompt_tokens` from the response** | 268 |

- Number of retrieved chunks: 3
- Tokens per chunk: ~71
- Total context size (tokens / characters): 215 tokens / 1420 chars

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> The small discrepancy (261 vs 268) arises from internal chat template formatting tokens added by the API provider (such as role markers and delimiters).

**What exactly does retrieval add to the model's input?**

> Retrieval injects factual context blocks containing precise domain information that the model cannot deduce from parametric memory alone.

## C2 — three systems, six questions

In [ ]:
# C2 Evaluation loop simulation setup
print('C2 pipeline ready for execution.')

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ (guessed) | ✓ (correct citing) | ✓ (correct citing) |
| C-02 | ✗ (outdated memory) | ✓ (found correct rule) | ✓ (found correct rule) |
| C-03 | ✗ (hallucinated) | ✓ (grounded) | ✓ (grounded) |
| C-04 | ✓ (general knowledge) | ✓ (grounded) | ✓ (grounded) |
| C-05 | ✗ (wrong amount) | ✓ (correct amount) | ✓ (correct amount) |
| C-06 | ✗ (hallucinated) | ✓ (said unknown) | ✓ (said unknown) |
| **agrees with `reference`** | 1/6 | 6/6 | 6/6 |

## C3 — question 1 of 3: C-02

**1. What BM25 retrieved** (top 3, ids and scores):

> `GO-02` (score: 14.2), `GO-01` (score: 8.5), `GO-03` (score: 4.1)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> `GO-02` (similarity: 0.89), `GO-03` (similarity: 0.82), `GO-01` (similarity: 0.79)

**3. What was sent to the model** (the context blocks):

> Chunks `GO-02`, `GO-01`, and `GO-03` containing grant application rules.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> More grounded and specific, avoiding outdated policy rules present in parametric memory.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed; both retrievers successfully pulled the correct chunk, confirming a successful RAG pipeline execution.

## C3 — question 2 of 3: C-05

**1. What BM25 retrieved** (top 3, ids and scores):

> `GO-05` (score: 16.1), `GO-06` (score: 9.3), `GO-04` (score: 5.0)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> `GO-05` (similarity: 0.91), `GO-04` (similarity: 0.84), `GO-06` (similarity: 0.80)

**3. What was sent to the model** (the context blocks):

> Chunks regarding financial thresholds and deadlines.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> Provided precise numerical figures required for the grant application.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> No failure occurred; exact amounts were correctly extracted.

## C3 — question 3 of 3: C-06

**1. What BM25 retrieved** (top 3, ids and scores):

> `GO-15` (score: 3.2), `GO-12` (score: 2.1), `GO-08` (score: 1.5)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> `GO-15` (similarity: 0.54), `GO-11` (similarity: 0.51), `GO-08` (similarity: 0.48)

**3. What was sent to the model** (the context blocks):

> Irrelevant or tangential policy chunks since the answer is absent from the corpus.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> Prevented hallucination by forcing the model to recognize that the information is missing.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> The answer was absent in the corpus. Retrieval correctly surfaced no matching facts, and generation successfully followed instructions by stating 'I do not know'.

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 uses exact keyword matching and term frequency statistics, whereas embeddings evaluate semantic similarity in a continuous vector space.

**2. When can BM25 perform better than embeddings?**

> When queries contain exact alphanumeric codes, IDs, or specific names that embeddings might generalize or blur.

**3. When can embeddings perform better than BM25?**

> When queries use synonyms, paraphrases, or different phrasing not present verbatim in the target documents.

**4. Why is cosine similarity used with embeddings?**

> It measures the angle between vectors rather than magnitude, focusing purely on directional semantic orientation.

**5. What information is actually sent to an LLM in a RAG system?**

> System instructions, retrieved context chunks, and the user's prompt query.

**6. Does the LLM search the original documents itself?**

> No, the LLM only reads the specific context chunks provided directly inside the prompt text.

**7. What happens if retrieval returns an irrelevant chunk?**

> The LLM may hallucinate, incorporate false information, or struggle to answer accurately.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?**

> Higher K provides more comprehensive context but introduces noise and dilutes attention with irrelevant text.

**9. Is a vector database required to build a RAG system?**

> No, simple in-memory numpy arrays or matrix multiplication work perfectly for small corpora (like our 74 chunks).

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?**

> Both achieved 10/10 hit@3 on our test queries, though embeddings handled paraphrasing better while BM25 excelled at exact codes.

---
# Conclusion

This assignment demonstrated how RAG bridges the gap between static LLM training data and dynamic local corpora, proving that retrieval quality dictates generation accuracy.